# ResNet18 ex novo v3.2.3 - Two-Phase + Reject Option

Variante v3.2.3: v3.1 (course-only) + Two-Phase Training (S:12) + Reject Option (Vento et al., S:11).

**Risultati v3.1 (0.9732 BA):**
- Single-phase full fine-tuning, CrossEntropy + class weights + LS=0.1
- Augmentation moderata, FP32 puro

**Novita v3.2.3 rispetto a v3.1:**
- **Two-Phase Training (S:12)**: freeze backbone 5 epoche (solo classificatore si adatta), poi unfreeze e fine-tuning completo
- **Reject Option (Vento et al., S:11, L:reject_option) - soglia scelta provando tutti i tau e massimizzando accepted BA con coverage >= 95%**:
  - \u03c8_a = max softmax confidence
  - \u03c8_b = top-2 margin
  - Val set splittato 50/50: val_cal per soglia, val_eval per metrica
  - Funzione di efficacia \u01a4 con Cc=1, Cr=3, Ce=10
- **Nessuna tecnica esterna al corso**: no Focal Loss, no AMP, no TTA, augmentation forte (S:05, L:04)

Obiettivo: sfruttare la two-phase (transfer learning da corso) per migliorare la convergenza del backbone, mantenendo la reject option rigorosa.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import datetime
import subprocess
from getpass import getpass

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/resnet18-exnovo-v323'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-resnet18-exnovo-v323-{RUN_ID}'
RUN_NAME = f'resnet18_exnovo_v323_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(sanitize(result.stdout))
    if result.stderr:
        print(sanitize(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    run(['git', 'clone', '--branch', BASE_BRANCH, AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime.')

run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', BASE_BRANCH], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
branch_exists = subprocess.run(['git', 'rev-parse', '--verify', RUN_BRANCH], cwd=REPO_PATH, text=True, capture_output=True).returncode == 0
if branch_exists:
    run(['git', 'switch', RUN_BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Run branch:', RUN_BRANCH)
print('Run name:', RUN_NAME)

## 2. Configurazione esperimento

In [ ]:
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP_NAME = 'resnet18_exnovo_v323'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.20
EPOCHS = 40
PATIENCE = 8
TWO_PHASE_FREEZE_EPOCHS = 5
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
DROPOUT_P = 0.25
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1



DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'

RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ProjectML_checkpoints') / RUN_NAME

CLASS_NAMES = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']
LABEL_MAP = {
    'battery': 0,
    'clothes': 1, 'clothing': 1, 'shoes': 1,
    'brown': 2, 'green': 2, 'transparent': 2, 'glass': 2,
    'metal': 3,
    'organic': 4,
    'cardboard': 5, 'paper': 5, 'papery': 5,
    'plastic': 6,
    'undifferentiated': 7,
}
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Dataset e split 80/20 (train / val_cal / val_eval)

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)

if not local_dataset.exists():
    if drive_dataset.exists():
        print('Copio dataset gia estratto da Drive a disco locale Colab...')
        shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        print('Estraggo zip dataset da Drive nella cartella locale attesa...')
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dataset)
    else:
        raise FileNotFoundError(f'Non trovo ne {drive_dataset} ne {drive_zip}')

if not local_dataset.exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and 'waste' in p.name.lower()]
    print('Candidate dataset folders:', candidates)
    raise FileNotFoundError(f'Dataset locale atteso non trovato: {local_dataset}')

print('Dataset locale:', local_dataset)

In [ ]:
samples = []
scanned_folders = set()
skipped_folders = set()
for dirpath, _, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if not filenames:
        continue
    scanned_folders.add(folder)
    if folder not in LABEL_MAP:
        skipped_folders.add(folder)
        continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            path = Path(dirpath) / fname
            samples.append({
                'path': str(path),
                'relative_path': str(path.relative_to(local_dataset)),
                'folder': folder,
                'label': label,
                'class_name': CLASS_NAMES[label],
            })

if skipped_folders:
    print(f'WARNING: {len(skipped_folders)} cartelle ignorate (non in LABEL_MAP): {sorted(skipped_folders)}')

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
if df.empty:
    raise RuntimeError('Nessuna immagine trovata. Controlla LABEL_MAP e struttura dataset.')

print('Totale immagini:', len(df))
print(df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0))

# Split 80/20 (train / val_all)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df['relative_path'], df['label']))
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'

# Split val 50/50 (val_cal / val_eval) stratificato
val_mask = df['split'] == 'val'
val_df_all = df[val_mask]
splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED+1)
cal_idx, eval_idx = next(splitter_val.split(val_df_all['relative_path'], val_df_all['label']))
val_indices = val_df_all.index
df.loc[val_indices[cal_idx], 'split'] = 'val_cal'
df.loc[val_indices[eval_idx], 'split'] = 'val_eval'

# Write split
split_path = RESULTS_DIR / 'split.csv'
df[['relative_path', 'folder', 'label', 'class_name', 'split']].to_csv(split_path, index=False)
print('Split salvato in:', split_path)
print('\nDistribuzione split:')
for s in ['train', 'val_cal', 'val_eval']:
    print(f'  {s}: {len(df[df["split"]==s])} samples')
print(pd.crosstab(df['class_name'], df['split']).reindex(CLASS_NAMES, fill_value=0))

## 4. Dataset, transform e DataLoader

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.ColorJitter(brightness=0.30, contrast=0.25, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_cal_df = df[df['split'] == 'val_cal'].reset_index(drop=True)
val_eval_df = df[df['split'] == 'val_eval'].reset_index(drop=True)

sample_counts = train_df['label'].value_counts().sort_index().to_numpy()
sample_inv_weights = 1.0 / np.maximum(sample_counts, 1.0)
train_sample_weights = sample_inv_weights[train_df['label'].values]
train_sampler = WeightedRandomSampler(train_sample_weights, num_samples=len(train_sample_weights), replacement=True)

train_loader = DataLoader(WasteDataset(train_df, train_transform), batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_cal_loader = DataLoader(WasteDataset(val_cal_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
val_eval_loader = DataLoader(WasteDataset(val_eval_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(dtype=np.float32)
class_weights = np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

weights_df = pd.DataFrame({'label': range(NUM_CLASSES), 'class_name': CLASS_NAMES, 'train_count': train_counts.astype(int), 'loss_weight': class_weights})
weights_df.to_csv(RESULTS_DIR / 'class_weights.csv', index=False)
print(weights_df)
print('WeightedRandomSampler attivo. Val split: val_cal / val_eval ready.')

## 5. Modello ResNet18

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(num_features, NUM_CLASSES),
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if not name.startswith('fc.')]
head_params = list(model.fc.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)

config = {
    'experiment': EXP_NAME,
    'run_id': RUN_ID,
    'run_name': RUN_NAME,
    'base_branch': BASE_BRANCH,
    'run_branch': RUN_BRANCH,
    'model': 'torchvision.models.resnet18 pretrained ImageNet',
    'strategy': 'two_phase_freeze5ep_strongAug_FP32_RejectOption_Vento_coverage95_psiA_psiB_split8020_val5050',
    'seed': SEED,
    'val_size': VAL_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'two_phase_freeze_epochs': TWO_PHASE_FREEZE_EPOCHS,
    'backbone_lr': BACKBONE_LR,
    'head_lr': HEAD_LR,
    'dropout_p': DROPOUT_P,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
        'reliability_functions': ['psi_a', 'psi_b'],
    'loss': 'CrossEntropyLoss with sqrt inverse frequency class weights and label smoothing 0.1',
    'augmentation': 'Resize, strong RandomResizedCrop (0.70-1.0), horizontal flip, 15deg rotation, affine shear/translate, strong ColorJitter, RandomErasing 25%; WeightedRandomSampler for class balancing',
    'course_arguments': 'transfer learning (two-phase freeze/unfreeze, S:12), CrossEntropy, label smoothing, ReduceLROnPlateau, early stopping, L2 regularization, dropout, data augmentation, class balancing (WeightedRandomSampler), balanced accuracy, Reject Option (Vento et al., S:11)',
    'dataset_drive_zip': DRIVE_ZIP_PATH,
    'dataset_drive_dir': DRIVE_DATASET_DIR,
    'results_dir': str(RESULTS_DIR),
    'checkpoint_drive_dir': str(DRIVE_CHECKPOINT_DIR),
}
with open(RESULTS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

In [ ]:
# Quick GPU memory check after model creation
if torch.cuda.is_available():
    print(f'GPU mem allocated: {torch.cuda.memory_allocated()/1024**3:.3f} GB')
    print(f'GPU mem reserved:  {torch.cuda.memory_reserved()/1024**3:.3f} GB')


## 6. Two-Phase Training con Strong Augmentation (su val_cal)

In [ ]:
def freeze_backbone(model, freeze):
    for name, p in model.named_parameters():
        if not name.startswith('fc.'):
            p.requires_grad = not freeze

def run_epoch(model, loader, train):
    model.train(train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in loader:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            if train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, targets)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_targets.append(targets.detach().cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()
    loss = total_loss / len(loader.dataset)
    acc = float((y_pred == y_true).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    return loss, acc, bal_acc, y_true, y_pred

best_ba = -1.0
best_ep = -1
no_impr = 0
history = []
best_ckpt = DRIVE_CHECKPOINT_DIR / 'resnet18_exnovo_v323_best.pth'

# Phase 1: backbone frozen, only classifier adapts
freeze_backbone(model, True)
print(f'Phase 1: backbone FROZEN ({TWO_PHASE_FREEZE_EPOCHS} epochs)')
for ep in range(1, TWO_PHASE_FREEZE_EPOCHS + 1):
    tl, ta, tba, _, _ = run_epoch(model, train_loader, True)
    vl, va, vba, _, _ = run_epoch(model, val_cal_loader, False)
    history.append({'epoch': ep, 'phase': 'freeze', 'train_loss': tl, 'train_ba': tba, 'val_loss': vl, 'val_ba': vba})
    print(f'  E{ep:02d}[freeze]: train_ba={tba:.4f} val_ba={vba:.4f}')
    if vba > best_ba:
        best_ba, best_ep = vba, ep
    scheduler.step(vba)

# Phase 2: backbone unfrozen, full fine-tuning
freeze_backbone(model, False)
no_impr = 0
print(f'Phase 2: backbone UNFROZEN')
for ep in range(TWO_PHASE_FREEZE_EPOCHS + 1, EPOCHS + 1):
    tl, ta, tba, _, _ = run_epoch(model, train_loader, True)
    vl, va, vba, _, _ = run_epoch(model, val_cal_loader, False)
    history.append({'epoch': ep, 'phase': 'finetune', 'train_loss': tl, 'train_ba': tba, 'val_loss': vl, 'val_ba': vba})
    lr_str = f'lr=[{optimizer.param_groups[0]["lr"]:.2e},{optimizer.param_groups[1]["lr"]:.2e}]'
    print(f'  E{ep:02d}[ftune]: {lr_str} train_ba={tba:.4f} val_ba={vba:.4f}')
    if vba > best_ba:
        best_ba, best_ep, no_impr = vba, ep, 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'config': config,
            'best_epoch': best_ep,
            'best_val_ba': best_ba,
        }, best_ckpt)
        print(f'    Saved best: BA={best_ba:.4f}')
    else:
        no_impr += 1
        if no_impr >= PATIENCE:
            print(f'Early stop epoch {ep}')
            break
    scheduler.step(vba)

pd.DataFrame(history).to_csv(RESULTS_DIR / 'history.csv', index=False)

summary = {
    'run_name': RUN_NAME,
    'run_branch': RUN_BRANCH,
    'results_dir': str(RESULTS_DIR),
    'best_epoch': best_ep,
    'best_val_ba': best_ba,
    'checkpoint_path_drive': str(best_ckpt),
}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nTraining complete. Best val_cal BA: {best_ba:.4f} at epoch {best_ep}')

## GPU Memory Check

In [ ]:
# GPU Memory Monitoring
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    max_alloc = torch.cuda.max_memory_allocated() / 1024**3
    print(f'GPU memory allocated: {allocated:.3f} GB')
    print(f'GPU memory reserved:  {reserved:.3f} GB')
    print(f'GPU max allocated:    {max_alloc:.3f} GB')
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU total memory:     {total:.3f} GB')
    print(f'GPU free:             {total - reserved:.3f} GB')
    # Check constraints (project: <4GB test, <5GB train)
    if max_alloc > 5.0:
        print(f'WARNING: max allocated ({max_alloc:.2f} GB) exceeds 5 GB training limit!')
    else:
        print(f'OK: max allocated ({max_alloc:.2f} GB) within 5 GB training limit.')
else:
    print('No GPU detected.')


## 7. Reject Option (Vento et al., S:11)

In [ ]:
checkpoint = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

def get_probs_and_labels(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(DEVICE)
            logits = model(images)
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(targets.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def find_optimal_threshold(psi, labels, probs, name='', min_coverage=0.95):
    preds = probs.argmax(axis=1)
    correct, error = (preds == labels), (preds != labels)
    N = len(labels)
    best_ba, best_tau, best_m = -1.0, 0.0, None
    # Sort thresholds descending -> coverage increases as tau decreases
    for tau in sorted(np.unique(psi), reverse=True):
        rejected = psi < tau
        accepted = ~rejected
        coverage = accepted.sum() / N
        if coverage < min_coverage:
            continue  # not enough samples accepted
        ba = float(balanced_accuracy_score(labels[accepted], preds[accepted]))
        if ba > best_ba:
            best_ba, best_tau = ba, tau
            best_m = {
                'threshold': float(tau),
                'reject_rate': float(1 - coverage),
                'coverage': float(coverage),
                'accepted_acc': float(correct[accepted].sum() / accepted.sum()),
                'accepted_ba': ba,
                'n_accepted': int(accepted.sum()),
                'n_rejected': int(rejected.sum()),
                'errors_avoided': int(error[rejected].sum()),
            }
    print(f"  {name}: tau*={best_m['threshold']:.4f} | reject={best_m['reject_rate']:.1%} | "
          f"acc_BA={best_m['accepted_ba']:.4f} | acc={best_m['accepted_acc']:.4f} | err_avoided={best_m['errors_avoided']}")
    return best_tau, best_m

# Step 1: calibrate thresholds on val_cal
print('--- Val_cal: calcolo probabilita e soglie ottimali ---')
probs_cal, labels_cal = get_probs_and_labels(model, val_cal_loader)
sorted_cal = np.sort(probs_cal, axis=1)[:, ::-1]
psi_a_cal = sorted_cal[:, 0]
psi_b_cal = sorted_cal[:, 0] - sorted_cal[:, 1]

tau_a, _ = find_optimal_threshold(psi_a_cal, labels_cal, probs_cal, 'psi_a')
tau_b, _ = find_optimal_threshold(psi_b_cal, labels_cal, probs_cal, 'psi_b')

# Step 2: evaluate on val_eval (unseen data)
print('\n--- Val_eval: applicazione soglie ---')
probs_eval, labels_eval = get_probs_and_labels(model, val_eval_loader)
sorted_eval = np.sort(probs_eval, axis=1)[:, ::-1]
preds_eval = probs_eval.argmax(axis=1)

# Baseline (no reject)
base_ba = float(balanced_accuracy_score(labels_eval, preds_eval))
base_acc = float((preds_eval == labels_eval).mean())
base_err = int((preds_eval != labels_eval).sum())
print(f'  No reject: BA={base_ba:.4f} Acc={base_acc:.4f} Errors={base_err}')

# With reject
reject_summary = {}
for name, tau, psi in [('psi_a', tau_a, sorted_eval[:, 0]), ('psi_b', tau_b, sorted_eval[:, 0] - sorted_eval[:, 1])]:
    rejected = psi < tau
    accepted = ~rejected
    ba = float(balanced_accuracy_score(labels_eval[accepted], preds_eval[accepted])) if accepted.sum() > 0 else 0.0
    acc = float((preds_eval[accepted] == labels_eval[accepted]).mean()) if accepted.sum() > 0 else 0.0
    cov = float(accepted.sum() / len(labels_eval))
    err_av = int((preds_eval[rejected] != labels_eval[rejected]).sum())
    print(f'  {name}: tau={tau:.4f} | reject={1-cov:.1%} | BA={ba:.4f} | Acc={acc:.4f} | err_avoided={err_av}/{base_err}')
    reject_summary[name] = {
        'threshold': float(tau),
        'reject_rate': float(1 - cov),
        'coverage': float(cov),
        'accepted_ba': ba,
        'accepted_acc': acc,
        'errors_avoided': err_av,
    }

with open(RESULTS_DIR / 'reject_standard.json', 'w') as f:
    json.dump(reject_summary, f, indent=2)

## 8. Metriche finali e grafici

In [ ]:
# Classification report on val_eval
report = classification_report(labels_eval, preds_eval, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(labels_eval, preds_eval, labels=list(range(NUM_CLASSES)))

with open(RESULTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR / 'confusion_matrix.csv')

print(report)
print('Final val_eval accuracy:', base_acc)
print('Final val_eval balanced accuracy:', base_ba)

# Update summary with reject results
summary['val_eval_ba'] = base_ba
summary['val_eval_acc'] = base_acc
summary['reject_standard'] = reject_summary
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Training curves (two-phase: freeze + finetune)
history_df = pd.read_csv(RESULTS_DIR / 'history.csv')
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for phase, c in [('freeze', 'orange'), ('finetune', 'blue')]:
    ph = history_df[history_df['phase'] == phase]
    if len(ph) > 0:
        plt.plot(ph['epoch'], ph['train_loss'], color=c, alpha=0.5, label=f'train {phase}')
        plt.plot(ph['epoch'], ph['val_loss'], color=c, ls='--', label=f'val_cal {phase}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet18 v3.2.3 Loss')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
for phase, c in [('freeze', 'orange'), ('finetune', 'blue')]:
    ph = history_df[history_df['phase'] == phase]
    if len(ph) > 0:
        plt.plot(ph['epoch'], ph['train_ba'], color=c, alpha=0.5, label=f'train {phase}')
        plt.plot(ph['epoch'], ph['val_ba'], color=c, ls='--', label=f'val_cal {phase}')
plt.xlabel('Epoch')
plt.ylabel('Balanced accuracy')
plt.title('ResNet18 v3.2.3 BA')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=160)
plt.show()

# Confusion matrix
plt.figure(figsize=(8, 7))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('ResNet18 v3.2.3 Confusion Matrix')
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=8, color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

# Normalized confusion matrix
cm_norm = cm.astype('float64') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, interpolation='nearest', cmap='Greens', vmin=0, vmax=1)
plt.title('ResNet18 v3.2.3 CM (normalized)')
plt.colorbar()
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_norm.png', dpi=160)
plt.show()

## 9. Note esperimento

In [ ]:
notes = f'''# {RUN_NAME}

## Ipotesi
La two-phase training (S:12) protegge il backbone pre-addestrato durante le prime epoche, permettendo al classificatore di convergere senza degradare i pesi ImageNet. Poi l'unfreeze completa il fine-tuning con learning rate differenziati. La Reject Option corregge gli errori residui.

## Modifiche v3.2.3 rispetto a v3.1
- AGGIUNTA Two-Phase Training (S:12): freeze backbone 5 epoche, poi unfreeze e fine-tuning completo
- AGGIUNTA Reject Option (Vento et al., S:11, L:reject_option)
- AGGIUNTO split val in val_cal (50% calibrazione) e val_eval (50% valutazione)
- AUMENTATE epoche da 35 a 40 per compensare le 5 epoche di sola testa
- RAFFORZATA data augmentation (scale 0.70, rot 15deg, affine shear/translate, jitter spinto, RandomErasing 25%) per ridurre gap train/val (S:05, L:04)

## Tecniche del corso
- Transfer learning / Two-Phase Training: S:12 (teoria) + L:08 (AlexNet fine-tuning)
- ResNet18 / Skip connections: S:14 (Residual Learning)
- StratifiedShuffleSplit: L:01 (KNN validation)
- Data augmentation (flip, rotazione, ColorJitter, RandomErasing): S:05 + L:04
- AdamW optimizer: S:09 (Adam) + L:04
- CrossEntropyLoss con class weights: S:09 + L:04
- Label smoothing: S:09 (regolarizzazione loss)
- WeightedRandomSampler: S:04-05 (class imbalance)
- ReduceLROnPlateau: S:09 (learning rate decay)
- Early stopping: S:09 + L:04
- Dropout: S:13 + L:07A
- L2 regularization (weight_decay): S:09
- Balanced accuracy: S:05
- Discriminative learning rates: S:12
- ImageNet normalization: L:07A
- Reject Option (Vento et al.): S:11 + L:reject_option

## Cosa NON c'e in v3.2.3
- Niente Focal Loss (non nel corso)
- Niente AMP (FP32 puro)
- Niente TTA (non richiesto per reject option)
- Niente torch.backends.cudnn.benchmark
- Niente non_blocking=True

## Risultato
Best epoch: {best_ep}
Best validation BA (val_cal): {best_ba:.4f}
Val_eval BA (no reject): {base_ba:.4f}
Branch run: {RUN_BRANCH}
Cartella risultati: {RESULTS_DIR}
Checkpoint Drive: {best_ckpt}

## File prodotti in repo
- `split.csv`
- `config.json`
- `class_weights.csv`
- `history.csv`
- `summary.json`
- `classification_report.txt`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `confusion_matrix_norm.png`
- `training_curves.png`
- `reject_standard.json`
'''
with open(RESULTS_DIR / 'notes.md', 'w') as f:
    f.write(notes)
print(notes)

## 10. Commit e push risultati su GitHub

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)

run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH)), 'src/resnet18_exnovo/resnet18_exnovo_v323_colab.ipynb'], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()

if not status:
    print('Nessuna modifica da committare.')
else:
    message = f'Add ResNet18 exnovo v3.2.3 results {RUN_ID}'
    run(['git', 'commit', '-m', message], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)

print('Risultati sincronizzati su GitHub.')